In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Day 2 - Data Exploration

Loading the CAlifornia Housing dataset and getting a first look at it.

Goal : Understand what features the dataset has, how many rows, and whether there are any data quality issues.

In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()

In [ ]:
df = pd.DataFrame(data.data, columns=data.feature_names)
df['Price'] = data.target

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

df.describe()

In [ ]:
df.isnull().sum()

## Day 2 - What I Learned

1)how to import california housing dataset

2)And arranged raw data into table form for easy to readable

# About DataSet
* 20,640 rows, 9 columns
* all columns are numeric(float64)
* And No errors in dataset

# Features
* MedInc - median income
* HouseAge - average house age
* AveRooms, AveBedrms - average rooms and bedrooms
* Population, AveOccup - population data
* Latitude, Logitude- location

# Things I noticed:
* Max value is 5.0
* Income(MedInc) varies a lot - 0.5 - 5.0

Day 3 - Data Visulation

Make plots to see patterns in data . numbers tells you facts:
plots tell yiu stories


In [ ]:


plt.figure(figsize=(10, 5))
plt.hist(df['Price'], bins=50)
plt.xlabel('Price (in $100,000s)')
plt.ylabel('Number of districts')
plt.title('Distribution of House Prices')
plt.show()

In [ ]:
df.hist(figsize=(15, 10), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation between features')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['MedInc'], df['Price'], alpha=0.1)
plt.xlabel('Median Income')
plt.ylabel("Price (in 100,000s)")
plt.title('Income vs Price')
plt.show()

In [ ]:
from scipy.__config__ import show
plt.figure(figsize=(10, 8))
plt.scatter(df['Longitude'], df['Latitude'],
            c=df['Price'], cmap='viridis', alpha=0.4, s=5)
plt.colorbar(label='Price (in $100,000s)')
plt.xlabel('Longtitude')
plt.ylabel('Latitude')
plt.title('House Prices across california')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

for i, feature in enumerate(features):
  row = i // 4
  col = i % 4
  axes[row, col].boxplot(df[feature])
  axes[row, col].set_title(feature)

plt.tight_layout()
plt.show()


## Day 3 what i have learneed from the plots

About Price:
  1. Most of the districts have price between 100k to 300k.
  2. Big spike at 500k mean too expensive but they mislabled data.
  3. My model will struggle with expensive houses

About Relationship with data (From heatmap):
  1. MedInc has the highest corrletion with price(~0.69).
  2. so Income matters more than I thought

About Geography
  1. by the Way I get to know coast are expensive
  2. inland areaas are cheaper
  3. loaction matters

About Data quality
  1. AVerooms, AveBedrooms, AveOccp have major outliers
  2. will handle later

Feature to matter:
  1. MedInc - confirmed by correlation
  2. Latitude / Longitude - Confirmed by California map





## Data Findings

### Top 3 Features Correlated with Price
1. **MedInc (~0.69)** — Strongest signal by far. Wealthier neighborhoods = higher prices. Makes intuitive sense.
2. **Latitude (~0.14)** — Northern parts of CA tend to be cheaper; Bay Area is north but coastal.
3. **AveRooms (~0.15)** — More rooms generally means a more expensive home.

Longitude and Latitude together paint the full geographic picture — they should be treated as a pair.

### Data Quality Issues

**Problem 1: $500K price cap**
- The dataset artificially caps house values at $500,001 (~5.0 in the scaled column).
- ~965 rows (~4.7%) hit this ceiling.
- **Impact:** Our model will underpredict for truly expensive homes because the training data never shows prices above $500K. We'll document this limitation and decide in Phase 3 whether to drop these rows or keep them.

**Problem 2: Outliers in AveRooms and AveOccup**
- Most districts have 4–8 rooms on average, but a few spike to 50+. These are likely hotels, dorms, or data entry errors for tiny census blocks.
- AveOccup has extreme values too (1000+ people per household) — clearly bad data for small blocks.
- **Impact:** These outliers will skew linear models. We'll cap or remove them in Phase 3.

### Features Expected to Matter Most
| Feature | Why |
|---|---|
| MedInc | Strongest correlation (0.69) |
| Latitude + Longitude | Location drives California prices heavily |
| AveRooms | Larger homes cost more |
| HouseAge | Older homes may be cheaper, but location complicates this |

### What I Noticed
- California housing is heavily driven by **where** and **how rich** — two features (location + income) carry most of the signal.
- The $500K cap is the biggest data quality issue and will affect model performance on expensive homes.
- AveRooms/AveOccup outliers are small in number but extreme in value — they'll need to be handled before modeling.

In [ ]:
# Step 3: Investigate outliers in AveRooms and AveOccup
print("=== AveRooms stats ===")
print(df['AveRooms'].describe())
print(f"\nRows with AveRooms > 50: {len(df[df['AveRooms'] > 50])}")

print("\n=== AveOccup stats ===")
print(df['AveOccup'].describe())
print(f"\nRows with AveOccup > 20: {len(df[df['AveOccup'] > 20])}")

# Show the extreme outliers
print("\n=== Worst AveRooms outliers ===")
print(df.nlargest(5, 'AveRooms')[['AveRooms', 'AveOccup', 'Population', 'Price']])

In [ ]:
# Step 2: Investigate the $500K price cap
# Price is in $100,000s, so 5.0 = $500,000
capped_rows = df[df['Price'] >= 5.0]
print(f"Total rows with Price >= $500K: {len(capped_rows)}")
print(f"That's {len(capped_rows)/len(df)*100:.1f}% of the dataset")
print()

# Visualize the cap
plt.figure(figsize=(10, 4))
plt.hist(df['Price'], bins=50, color='steelblue', edgecolor='white')
plt.axvline(x=5.0, color='red', linestyle='--', linewidth=2, label='$500K cap')
plt.xlabel('Price (in $100,000s)')
plt.ylabel('Count')
plt.title('Price Distribution — notice the spike at $500K (artificial cap)')
plt.legend()
plt.show()

In [ ]:
# Step 1: Top 3 features most correlated with Price
correlations = df.corr()['Price'].drop('Price').sort_values(ascending=False)
print("All feature correlations with Price:\n")
print(correlations)
print("\n--- Top 3 ---")
print(correlations.head(3))

## Day 4 - Data Analysis & Findings

Goal: Identify the top correlated features, spot data quality problems, and write a clear summary of what we found.